<a href="https://colab.research.google.com/github/mushynskaaa/ab-test-significance/blob/main/ab-test-significance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Connecting Libraries and Importing a Dataset**

In [ ]:
# Connecting libraries
import pandas as pd
import numpy as np
from scipy.stats import norm
from google.colab import drive

# Connecting to google drive and importing a dataset
drive.mount("/content/drive")

df = pd.read_csv("/content/drive/MyDrive/Mate_Academy/portfolio_2/dataset_for_portfolio2.csv")

# Print the first 5 lines to verify that the data was transferred correctly
df.head()

Mounted at /content/drive


,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-03,Qatar,mobile,Asia,Organic Search,2,2,new account,1
1,2020-11-03,Ecuador,mobile,Americas,Direct,2,2,new account,1
2,2020-11-12,New Zealand,mobile,Oceania,Undefined,2,2,new account,1
3,2020-11-12,Bulgaria,mobile,Europe,Paid Search,2,2,new account,1
4,2020-11-15,Bulgaria,desktop,Europe,Social Search,2,2,new account,1


`date` - session date.

`country` - user’s country based on IP.

`device` - device type (desktop / mobile / tablet).

`continent` - user’s continent.

`channel`- traffic channel (may have the value “Undefined” this is a category, not a missing value).

`test` - A/B test sequence number.

`test_group` - group in the test (1 = A / control, 2 = B / variant).

`event_name` - name of the event metric; specifies what the `value` column counts.

`value` - the quantity of what is specified in `event_name` for this segment.

## **Data Overview**

In [ ]:
# Data type checking
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  object
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  int64 
 6   test_group  800996 non-null  int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 55.0+ MB


In [ ]:
# Changing the data type of a date column
df["date"] = pd.to_datetime(df["date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        800996 non-null  datetime64[ns]
 1   country     800996 non-null  object        
 2   device      800996 non-null  object        
 3   continent   800996 non-null  object        
 4   channel     800996 non-null  object        
 5   test        800996 non-null  int64         
 6   test_group  800996 non-null  int64         
 7   event_name  800996 non-null  object        
 8   value       800996 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(5)
memory usage: 55.0+ MB


In [ ]:
# Checking for missing values by column
df.isna().sum()

,0
date,0
country,0
device,0
continent,0
channel,0
test,0
test_group,0
event_name,0
value,0


There are no missing values.

## **Statistical Significance Calculation**
### **Overall test score**

In [ ]:
# Creating variables for subsequent calculation of metrics, as well as filtering data.
key_metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new account"]
denominator = "session"

significance_data = df[df["event_name"].isin(key_metrics + [denominator])].copy()

significance_data["event_name"].unique()

array(['new account', 'session', 'begin_checkout', 'add_payment_info',
       'add_shipping_info'], dtype=object)

In [ ]:
# Calculating the Z-Test
from statsmodels.stats.proportion import proportions_ztest

def calculate_significance(num_control, den_control, num_test, den_test):
    """
    Two-proportions z-test for a single metric.
    Input: numerator/denominator of the control group (Group 1) and the test group (Group 2).
    Output: conversion rates (%), relative change (%), z-statistic, p-value.
    """

# Calculating conversions
    rate_control = num_control / den_control
    rate_test = num_test / den_test

# Relative change
    metric_change = (rate_test - rate_control) / rate_control * 100

# "nobs" = number of observations
    z_stat, p_value = proportions_ztest(
      count=[num_test, num_control],
      nobs=[den_test, den_control]
    )

    return rate_control * 100, rate_test * 100, metric_change, z_stat, p_value

In [ ]:
def significance_table(data):
  """
  Runs through all tests and metrics, calculates the coverage, and collects the results in a DataFrame.
  """
  results = []

  for test_number in sorted(data["test"].unique()):
        test_data = data[data["test"] == test_number]

        control = test_data[test_data["test_group"] == 1]
        test = test_data[test_data["test_group"] == 2]

        den_control = control[control["event_name"] == denominator]["value"].sum()
        den_test = test[test["event_name"] == denominator]["value"].sum()

        for metric in key_metrics:
            num_control = control[control["event_name"] == metric]["value"].sum()
            num_test = test[test["event_name"] == metric]["value"].sum()

            rate_c, rate_t, change, z_stat, p_value = calculate_significance(num_control, den_control, num_test, den_test)

            results.append({
                "test_number": test_number,
                "metric": f"{metric}/{denominator}",
                "numerator_event": metric,
                "denominator_event": denominator,
                "numerator_control": num_control,
                "denominator_control": den_control,
                "conversion_rate_control": rate_c,
                "numerator_test": num_test,
                "denominator_test": den_test,
                "conversion_rate_test": rate_t,
                "metric_change": change,
                "z_stat": z_stat,
                "p_value": p_value,
                "significant": p_value < 0.05
            })

  return pd.DataFrame(results)

In [ ]:
# Run for calculations
significance_results = significance_table(significance_data)

significance_results

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant
0,1,add_payment_info/session,add_payment_info,session,1988,45362,4.382523,2229,45193,4.932180,12.542021,3.924884,0.000087,True
1,1,add_shipping_info/session,add_shipping_info,session,3034,45362,6.688418,3221,45193,7.127210,6.560481,2.603571,0.009226,True
2,1,begin_checkout/session,begin_checkout,session,3784,45362,8.341784,4021,45193,8.897396,6.660587,2.978783,0.002894,True
3,1,new account/session,new account,session,3823,45362,8.427759,3681,45193,8.145067,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info/session,add_payment_info,session,2344,50637,4.629026,2409,50244,4.794602,3.576911,1.240994,0.214608,False
5,2,add_shipping_info/session,add_shipping_info,session,3480,50637,6.872445,3510,50244,6.985909,1.650995,0.709557,0.477979,False
6,2,begin_checkout/session,begin_checkout,session,4262,50637,8.416770,4313,50244,8.584110,1.988164,0.952898,0.340642,False
7,2,new account/session,new account,session,4165,50637,8.225211,4184,50244,8.327362,1.241934,0.588793,0.556000,False
8,3,add_payment_info/session,add_payment_info,session,3623,70047,5.172241,3697,70439,5.248513,1.474630,0.643172,0.520112,False
9,3,add_shipping_info/session,add_shipping_info,session,5298,70047,7.563493,5188,70439,7.365238,-2.621211,-1.413727,0.157442,False


### **Conclusions**
**Test 1** - is the only one with a statistically significant increase in results. The new version significantly boosted all key metrics: `add_payment_info` (+12.5%), `begin_checkout` (+6.7%), and `add_shipping_info` (+6.6%) and all of these increases are statistically significant. Sign-ups remained unchanged; changes clearly need to be implemented.

**Test 2** – nothing statistically significant happened. The metrics showed changes within the range of 1–3%, which is an insignificant result. The test version is no worse, but also no better than the previous one.

**Test 3** - there is already a decline here. `begin_checkout` dropped significantly by 3.4%. The rest of the metrics are not statistically significant, but they also show negative trends. These changes should not be implemented.

**Test 4** - this one has the worst results. It has the largest sample size (105k), so even small changes are statistically significant, and almost all are negative: `begin_checkout` (−2.4%), `new account` (−3.4%). This test must be stopped immediately.

**Conclusion:** Of the four tests, only the first one should be implemented. Test 4 is definitely a no-go, test 2 is neutral and test 3 is mostly negative. For the latter three, additional analysis should be conducted to identify the causes of this behavior and find an alternative, if possible.

In [ ]:
# Saving data for visualization in Tableau
from google.colab import drive

drive.mount("/content/drive")

significance_results.to_csv("/content/drive/MyDrive/ab_test_significance_total.csv", index=False)

print("Saved")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved


### **Building a universal function for calculating metrics in specific segments**

In [ ]:
def significance_table(data, breakdown=None):
  """
  Calculates the significance for each test and metric.
  breakdown=None will calculate the total for the test (as before).
  breakdown=[“device” (or any other segment)] will calculate the same total, but separately for each device.
  """
  breakdown = breakdown or []
  group_cols = ["test"] + breakdown
  results = []

  for keys, unit in data.groupby(group_cols):
        keys = keys if isinstance(keys, tuple) else (keys,)
        key_map = dict(zip(group_cols, keys))

        control = unit[unit["test_group"] == 1]
        test = unit[unit["test_group"] == 2]

        den_control = control[control["event_name"] == denominator]["value"].sum()
        den_test = test[test["event_name"] == denominator]["value"].sum()

        if den_control == 0 or den_test == 0:
            continue

        for metric in key_metrics:
            num_control = control[control["event_name"] == metric]["value"].sum()
            num_test = test[test["event_name"] == metric]["value"].sum()

            rate_c, rate_t, change, z_stat, p_value = calculate_significance(num_control, den_control, num_test, den_test)

            row = {"test_number": key_map["test"]}
            for b in breakdown:
                row[b] = key_map[b]
            row.update({
                "metric": f"{metric}/{denominator}",
                "numerator_event": metric,
                "denominator_event": denominator,
                "numerator_control": num_control,
                "denominator_control": den_control,
                "conversion_rate_control": rate_c,
                "numerator_test": num_test,
                "denominator_test": den_test,
                "conversion_rate_test": rate_t,
                "metric_change": change,
                "z_stat": z_stat,
                "p_value": p_value,
                "significant": p_value < 0.05,
            })
            results.append(row)

  return pd.DataFrame(results)

### **Breakdown by device**

In [ ]:
significance_by_device = significance_table(significance_data, breakdown=["device"])
significance_by_device

,test_number,device,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant
0,1,desktop,add_payment_info/session,add_payment_info,session,1130,26467,4.269468,1256,26417,4.754514,11.360819,2.686998,0.007210,True
1,1,desktop,add_shipping_info/session,add_shipping_info,session,1711,26467,6.464654,1916,26417,7.252905,12.193247,3.586024,0.000336,True
2,1,desktop,begin_checkout/session,begin_checkout,session,2108,26467,7.964635,2404,26417,9.100201,14.257595,4.673980,0.000003,True
3,1,desktop,new account/session,new account,session,2203,26467,8.323573,2147,26417,8.127342,-2.357527,-0.821212,0.411526,False
4,1,mobile,add_payment_info/session,add_payment_info,session,810,17896,4.526151,942,17767,5.301964,17.140683,3.389330,0.000701,True
5,1,mobile,add_shipping_info/session,add_shipping_info,session,1257,17896,7.023916,1256,17767,7.069286,0.645933,0.167387,0.867065,False
6,1,mobile,begin_checkout/session,begin_checkout,session,1593,17896,8.901430,1561,17767,8.785951,-1.297308,-0.384029,0.700957,False
7,1,mobile,new account/session,new account,session,1530,17896,8.549397,1441,17767,8.110542,-5.133163,-1.499486,0.133748,False
8,1,tablet,add_payment_info/session,add_payment_info,session,48,999,4.804805,31,1009,3.072349,-36.056739,-1.996608,0.045868,True
9,1,tablet,add_shipping_info/session,add_shipping_info,session,66,999,6.606607,49,1009,4.856293,-26.493378,-1.687725,0.091464,False


### **Conclusions**
Overall, most of the results were weak or insignificant, but a breakdown by device shows that this averaging masks more detailed results: the test effect is concentrated on specific platforms and is not uniform.

**Desktop** - has the largest sample size (26–61 thousand sessions per group) and the best results. In Test 1, the new version significantly increased `add_payment_info` (+11.4%), `add_shipping_info` (+12.2%), and `begin_checkout` (+14.3%). In Test 4, however, there was a significant decline across all metrics.

**Mobile** - metrics that are significant on desktop lose their significance on mobile (Test 1, `add_shipping`: +0.6%, p=0.87). A more detailed analysis is needed to determine the cause of this behavior and find an alternative for these users.

**Tablet** - the sample size is too small to draw accurate conclusions. This is likely why the results show extreme fluctuations (Test 4: −49% for `add_payment`, Test 1: −36%). These results should not be interpreted as a real effect, additional testing and analysis are also needed.

### **Breakdown by continent**

In [ ]:
significance_data = significance_data[significance_data["country"] != "(not set)"]
significance_by_continent = significance_table(significance_data, breakdown=["continent"])
significance_by_continent

,test_number,continent,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant
0,1,Africa,add_payment_info/session,add_payment_info,session,18,422,4.265403,19,395,4.810127,12.770745,0.374192,0.708262,False
1,1,Africa,add_shipping_info/session,add_shipping_info,session,37,422,8.767773,24,395,6.075949,-30.701334,-1.462805,0.143521,False
2,1,Africa,begin_checkout/session,begin_checkout,session,48,422,11.374408,29,395,7.341772,-35.453586,-1.971485,0.048668,True
3,1,Africa,new account/session,new account,session,33,422,7.819905,35,395,8.860759,13.310318,0.538221,0.590425,False
4,1,Americas,add_payment_info/session,add_payment_info,session,1104,25110,4.396655,1177,24713,4.762676,8.324984,1.954364,0.050658,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,4,Europe,new account/session,new account,session,1630,19572,8.328224,1620,19415,8.344064,0.190195,0.056571,0.954887,False
76,4,Oceania,add_payment_info/session,add_payment_info,session,75,1095,6.849315,38,1069,3.554724,-48.101029,-3.444324,0.000572,True
77,4,Oceania,add_shipping_info/session,add_shipping_info,session,91,1095,8.310502,59,1069,5.519177,-33.587928,-2.555993,0.010589,True
78,4,Oceania,begin_checkout/session,begin_checkout,session,218,1095,19.908676,137,1069,12.815716,-35.627483,-4.454701,0.000008,True


### **Conclusions**
In the large segments (Americas, Europe, Asia), the results are moderate and fully consistent with the total. However, in the smaller segments, Africa and Oceania, the sample size is only 400–1,100 sessions, this is likely why the z-test shows anomalous changes with apparent statistical significance. For example, in Test 4, `begin_checkout` changed by −2.4% for the total (105k sessions) and by −36% for Oceania (1k sessions). Although this is the same test, we should rely on the total the sample size is larger, and the margin of error is consequently smaller. It is worth expanding the sample size and analyzing the results in greater detail.

### **Important**
The analysis was conducted both at the aggregate level and across various dimensions (by device, etc.), but the dashboard displays only the difference at the aggregate level across tests: this directly answers the key question of A/B testing, whether there is a statistically significant effect in the test and whether it’s worth implementing.

The full set of breakdowns remains here in the notebook to demonstrate the tool’s scalability and serve as a basis for in-depth analysis (e.g., by desktop), but displaying all 48 breakdowns on the dashboard would overload it and, in all likelihood, distort the main conclusion.


## **Link to the dashboard and the file containing the total calculations (CSV)**

#### **Vizualization in Tableau:**
[Link to Dashboard](https://public.tableau.com/views/ABtestdashboard_17889006779050/ABtest?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

#### **File with calculations:**
[Link to file](https://drive.google.com/file/d/1O4mms8G36nZYSIxPwf_65ArcRBh84iMT/view?usp=sharing)